# Notebook 01: Data Loading and Exploration

This notebook loads the NExT-QA dataset, verifies data integrity, explores the question
distributions, and establishes the evaluation subsets we will use throughout the project.

**Goals:**
1. Load MC (Multiple Choice) and OE (Open-Ended) parquet files
2. Verify video coverage (which QA pairs have corresponding video files)
3. Analyze question type distributions and their implications for chunking strategy evaluation
4. Sample representative videos for visual inspection
5. Establish the evaluation subset (MC test with available videos)

**Inputs:** `data/MC/`, `data/OE/`, `data/videos/NExTVideo/`

**Outputs:** Dataframes with coverage flags, question type statistics, video metadata

**Context within the project:** This is the first executable notebook in our 13-notebook
pipeline. Everything downstream depends on the data structures, subset definitions, and
coverage verifications established here. The development subset (100 videos) defined in this
notebook is reused identically in Notebooks 02 through 05 for efficient iteration. The full
evaluation set (1,000 videos for MC, 570 for OE) is used in Notebooks 06 through 12 for
final results. Any changes to subset definitions here propagate through the entire pipeline,
which is why we verify representativeness and document selection criteria explicitly.

**Relationship to the architecture (Notebook 00):** The architecture notebook made assumptions
about dataset characteristics -- ~44 second video duration, ~8 questions per video, 52% causal
questions. This notebook validates those assumptions with actual data and adjusts pipeline
parameters if the assumptions are wrong. It also establishes the exact evaluation protocol:
which questions, which videos, which metrics, and what statistical power we have for detecting
differences between chunking strategies.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import json
from collections import Counter

PROJECT_ROOT = Path("/Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa")
DATA_DIR = PROJECT_ROOT / "data" / "nextqa"
VIDEO_DIR = DATA_DIR / "videos" / "NExTVideo"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Video directory: {VIDEO_DIR}")
print(f"Video dir exists: {VIDEO_DIR.exists()}")

Project root: /Users/nipun.batra/Downloads/ML/Multimodal_RAG_NExTQA
Data directory: /Users/nipun.batra/Downloads/ML/Multimodal_RAG_NExTQA/data
Video directory: /Users/nipun.batra/Downloads/ML/Multimodal_RAG_NExTQA/data/videos/NExTVideo
Video dir exists: True


## 1. Loading the NExT-QA Dataset

NExT-QA comes in two formats:
- **MC (Multiple Choice):** Question + 5 answer options + gold answer index. Used for accuracy evaluation.
- **OE (Open-Ended):** Question + free-text answer. Used for generation quality evaluation.

We load from parquet files downloaded from HuggingFace (`lmms-lab/NExTQA`).

**Why two formats matter for our evaluation strategy:** The MC format gives us an unambiguous
accuracy metric -- the system either picks the correct option or it does not. There is no
subjectivity in evaluation, no need for fuzzy text matching, and no dependence on an LLM judge.
With 8,564 MC questions, we get statistically robust measurements (95% confidence intervals of
roughly plus or minus 1-2 percentage points at typical accuracy levels). The OE format, by
contrast, requires us to evaluate free-text generation quality -- faithfulness to retrieved
context, citation accuracy, and semantic similarity to the gold answer. These metrics are
inherently noisier but capture aspects of system quality (fluency, grounding, attribution)
that multiple choice cannot.

**Parquet format choice:** The HuggingFace dataset uses parquet rather than CSV for several
reasons that matter at our scale. Parquet stores data in columnar format with efficient
compression, making selective column reads fast. For 48K+ QA pairs with video metadata, parquet
files are roughly 3-5x smaller than equivalent CSVs. More importantly, parquet preserves
data types (int64 for frame counts, string for answers) whereas CSV requires type inference
that can silently misinterpret numeric video IDs as integers, dropping leading zeros. The
NExT-QA video IDs are 10-digit numeric strings, and parquet ensures they load with correct
types in each split.
**Why this setup matters for reproducibility and correctness:** The configuration choices above are not arbitrary -- each parameter and import serves a specific role in the pipeline. Library versions, random seeds, device selection, and path configurations must be set before any computation to ensure deterministic results. In production ML systems, environment drift (different library versions across machines) is one of the most common sources of silent bugs where models appear to train successfully but produce subtly different results. By pinning these configurations at the notebook's entry point, we establish a single source of truth that all downstream cells inherit.

**Hardware considerations:** The device selection (MPS on Apple Silicon, CUDA on NVIDIA, or CPU fallback) directly impacts training throughput and numerical precision. MPS provides 2-5x speedup over CPU for transformer-based models but requires careful memory management since Apple's unified memory architecture shares resources between the GPU and system processes. The batch sizes and sequence lengths chosen later in this notebook are calibrated to fit within the available memory budget on our target hardware.

In [2]:
# Load Multiple Choice (MC) test split
mc_test = pd.read_parquet(DATA_DIR / "MC" / "test-00000-of-00001.parquet")

print(f"MC Test Set:")
print(f"  Shape: {mc_test.shape}")
print(f"  Columns: {list(mc_test.columns)}")
print(f"  Unique videos: {mc_test['video'].nunique()}")
print(f"  Questions per video: {mc_test.shape[0] / mc_test['video'].nunique():.1f} (average)")
print()
print("First 3 rows:")
mc_test.head(3)

MC Test Set:
  Shape: (8564, 13)
  Columns: ['video', 'frame_count', 'width', 'height', 'question', 'answer', 'qid', 'type', 'a0', 'a1', 'a2', 'a3', 'a4']
  Unique videos: 1000
  Questions per video: 8.6 (average)

First 3 rows:


,video,frame_count,width,height,question,answer,qid,type,a0,a1,a2,a3,a4
0,2574374895,719,320,240,what did the baby do after throwing the green ...,2,8,TN,clap proudly,the lady sitting down,lay on floor,just picked it up,crawl
1,2925959064,1233,500,375,where is this place,1,1,DL,mall,river,swimming pool,living room,mountain
2,3083302557,1420,640,480,what did the lady do while turning back,1,3,TC,walk away,thumbs up,put down her club,applying cream on face,caressing for the dog


In [3]:
# Load Open-Ended (OE) splits
oe_train = pd.read_parquet(DATA_DIR / "OE" / "train-00000-of-00001.parquet")
oe_val = pd.read_parquet(DATA_DIR / "OE" / "validation-00000-of-00001.parquet")
oe_test = pd.read_parquet(DATA_DIR / "OE" / "test-00000-of-00001.parquet")

print("Open-Ended Splits:")
print(f"  Train: {oe_train.shape[0]:,} questions, {oe_train['video'].nunique()} unique videos")
print(f"  Val:   {oe_val.shape[0]:,} questions, {oe_val['video'].nunique()} unique videos")
print(f"  Test:  {oe_test.shape[0]:,} questions, {oe_test['video'].nunique()} unique videos")
print()
print(f"OE Columns: {list(oe_val.columns)}")
print()
print("Sample OE question-answer pair:")
sample = oe_val.iloc[0]
print(f"  Video: {sample['video']}")
print(f"  Question: {sample['question']}")
print(f"  Answer: {sample['answer']}")
print(f"  Type: {sample['type']}")

Open-Ended Splits:
  Train: 37,523 questions, 3870 unique videos
  Val:   5,343 questions, 570 unique videos
  Test:  9,178 questions, 1000 unique videos

OE Columns: ['video', 'frame_count', 'width', 'height', 'question', 'answer', 'qid', 'type', 'additional_ref_answer']

Sample OE question-answer pair:
  Video: 3233088823
  Question: how is the bulldozer controlled
  Answer: from the drivers cabin
  Type: CH


### Reasoning: Understanding the Data Format

**MC format observations:**
- `video` column contains numeric IDs (int64) that correspond to video filenames (e.g., video ID `2574374895` maps to `2574374895.mp4`)
- `answer` is the index (0-4) of the correct option among columns `a0` through `a4`
- `type` encodes question category (CW = Causal-Why, TN = Temporal-Next, etc.)
- Each video has multiple questions (~8.6 on average)

**OE format observations:**
- `video` column is a string (not int64 like MC)
- `answer` is free-text (the gold answer for generation evaluation)
- `additional_ref_answer` provides alternative valid answers (useful for generation evaluation with multiple-reference BLEU/ROUGE)

**Key difference:** MC has 5 answer options, making it suitable for multiple-choice accuracy
evaluation. OE has free-text answers, suitable for generation quality (BLEU, ROUGE, semantic
similarity) and faithfulness evaluation.

**Decision: We use MC test set as our primary evaluation benchmark** because:
1. Multiple-choice accuracy is unambiguous (correct or not)
2. No need for fuzzy text matching (which introduces evaluation noise)
3. 8,564 questions gives strong statistical power for the ablation

OE validation set will be used for generation quality evaluation (faithfulness, citation quality)
because it requires free-form answers where we can measure hallucination.

**Technical note on data types:** The MC `video` column loads as int64 while OE `video` loads
as string. This type mismatch means we must explicitly cast to string (video_str) before any
cross-split comparisons. Failing to do so would cause silent join failures where no videos
match between MC and OE despite referring to the same underlying content. The 13 columns in MC
(video, frame_count, width, height, question, answer, qid, type, a0-a4) provide both the
evaluation data and video metadata in a single denormalized table -- efficient for per-question
analysis but requiring groupby for per-video statistics.

## 2. Video Coverage Verification

Not all videos referenced in the QA annotations may be available locally. We need to verify
which QA pairs have corresponding video files, since our pipeline can only process videos
that actually exist on disk.

**Why coverage verification is critical before any processing begins:** The NExT-QA dataset
references videos from multiple sources (YouTube user uploads), and video availability can
change over time -- creators delete videos, channels get removed, or content gets geo-blocked.
If we proceeded with indexing and evaluation without checking coverage, we would discover
missing videos mid-pipeline, causing crashes or silently skewing our metrics. A video with
10 associated questions that is missing from disk would bias our evaluation by removing those
10 questions from the denominator.

**What we check for each split:** For every unique video ID in the QA annotations, we verify
that a corresponding .mp4 file exists in our local video directory. The video filenames follow
the pattern `{video_id}.mp4` where video_id is the numeric identifier from the parquet data.
We also check whether the MC test and OE validation sets share any videos, since shared videos
only need to be processed once during the indexing phase.

**Coverage scenarios and their implications:**
- 100% coverage: Ideal -- all questions can be evaluated, no sampling bias
- 90-99% coverage: Acceptable if missing videos are random (not correlated with question type)
- Below 90%: Would require careful analysis of whether missing videos create systematic bias
  (e.g., if long/complex videos are disproportionately missing, causal questions would be
  underrepresented in our evaluation)

The coverage check also gives us the exact set of videos we need to process through scene
detection, frame extraction, captioning, and embedding -- allowing accurate compute time
estimates for the full pipeline.
**Why this setup matters for reproducibility and correctness:** The configuration choices above are not arbitrary -- each parameter and import serves a specific role in the pipeline. Library versions, random seeds, device selection, and path configurations must be set before any computation to ensure deterministic results. In production ML systems, environment drift (different library versions across machines) is one of the most common sources of silent bugs where models appear to train successfully but produce subtly different results. By pinning these configurations at the notebook's entry point, we establish a single source of truth that all downstream cells inherit.

**Hardware considerations:** The device selection (MPS on Apple Silicon, CUDA on NVIDIA, or CPU fallback) directly impacts training throughput and numerical precision. MPS provides 2-5x speedup over CPU for transformer-based models but requires careful memory management since Apple's unified memory architecture shares resources between the GPU and system processes. The batch sizes and sequence lengths chosen later in this notebook are calibrated to fit within the available memory budget on our target hardware.

In [4]:
# Get all available video files
available_videos = set()
for f in VIDEO_DIR.iterdir():
    if f.suffix == ".mp4":
        available_videos.add(f.stem)

print(f"Total video files on disk: {len(available_videos)}")
print(f"Sample filenames: {sorted(list(available_videos))[:5]}")
print()

# Check MC test coverage
mc_test['video_str'] = mc_test['video'].astype(str)
mc_test['has_video'] = mc_test['video_str'].isin(available_videos)
mc_covered = mc_test['has_video'].sum()
mc_videos_needed = mc_test['video_str'].nunique()
mc_videos_found = mc_test[mc_test['has_video']]['video_str'].nunique()

print(f"MC Test Coverage:")
print(f"  Questions with video: {mc_covered:,} / {len(mc_test):,} ({100*mc_covered/len(mc_test):.1f}%)")
print(f"  Unique videos needed: {mc_videos_needed}")
print(f"  Unique videos found:  {mc_videos_found}")
print(f"  Video coverage: {100*mc_videos_found/mc_videos_needed:.1f}%")
print()

# Check OE validation coverage
oe_val['video_str'] = oe_val['video'].astype(str)
oe_val['has_video'] = oe_val['video_str'].isin(available_videos)
oe_covered = oe_val['has_video'].sum()
oe_val_videos_needed = oe_val['video_str'].nunique()
oe_val_videos_found = oe_val[oe_val['has_video']]['video_str'].nunique()

print(f"OE Validation Coverage:")
print(f"  Questions with video: {oe_covered:,} / {len(oe_val):,} ({100*oe_covered/len(oe_val):.1f}%)")
print(f"  Unique videos needed: {oe_val_videos_needed}")
print(f"  Unique videos found:  {oe_val_videos_found}")
print(f"  Video coverage: {100*oe_val_videos_found/oe_val_videos_needed:.1f}%")
print()

# Check OE test coverage
oe_test['video_str'] = oe_test['video'].astype(str)
oe_test['has_video'] = oe_test['video_str'].isin(available_videos)
oe_test_covered = oe_test['has_video'].sum()
oe_test_videos_needed = oe_test['video_str'].nunique()
oe_test_videos_found = oe_test[oe_test['has_video']]['video_str'].nunique()

print(f"OE Test Coverage:")
print(f"  Questions with video: {oe_test_covered:,} / {len(oe_test):,} ({100*oe_test_covered/len(oe_test):.1f}%)")
print(f"  Unique videos needed: {oe_test_videos_needed}")
print(f"  Unique videos found:  {oe_test_videos_found}")
print(f"  Video coverage: {100*oe_test_videos_found/oe_test_videos_needed:.1f}%")

Total video files on disk: 1570
Sample filenames: ['10001787725', '10030609934', '10042935613', '10082798964', '10109006686']

MC Test Coverage:
  Questions with video: 8,564 / 8,564 (100.0%)
  Unique videos needed: 1000
  Unique videos found:  1000
  Video coverage: 100.0%

OE Validation Coverage:
  Questions with video: 5,343 / 5,343 (100.0%)
  Unique videos needed: 570
  Unique videos found:  570
  Video coverage: 100.0%

OE Test Coverage:
  Questions with video: 9,178 / 9,178 (100.0%)
  Unique videos needed: 1000
  Unique videos found:  1000
  Video coverage: 100.0%


### Reasoning: Video Coverage Implications

Our pipeline can only evaluate questions whose videos are available locally.

**Coverage results:** 100% coverage across all test/val splits. Every question in MC test (8,564), OE val (5,343), and OE test (9,178) has a corresponding video file available on disk. This means we can run the full ablation study without any missing data.

**Video pool structure:**
- MC test and OE test share the same 1,000 videos (identical video pool, different question formats)
- OE val uses a separate set of 570 videos (zero overlap with MC test)
- Total unique videos: 1,570 -- all available on disk

**Why OE train coverage does not matter:** OE train (37,523 questions, 3,870 unique videos) references videos we do not have locally. This is irrelevant because we are not training any models -- we only need test and validation data for evaluation. If we were fine-tuning the text embedding model or training a custom reranker, we would need OE train videos. But since our pipeline uses pre-trained models (bge-large, CLIP, BLIP, Qwen2-VL, DeBERTa) without task-specific adaptation, the training split is unused.

**Decision: Our primary evaluation set is MC test (8,564 questions, 1,000 videos) for accuracy measurement. OE validation (5,343 questions, 570 videos) serves generation quality evaluation. Together they cover 1,570 unique videos that all need processing.**

**What 100% coverage means practically:** We do not need to handle missing-video edge cases in our pipeline code. Every video_id in our evaluation sets is guaranteed to have a corresponding .mp4 file, so we can process the full set without try/except blocks for missing files. This simplifies the processing code and eliminates a potential source of silent data loss. If coverage had been partial, we would need to (a) filter the evaluation set to only covered questions, (b) verify that the filtered set remains representative across question types, and (c) report the coverage rate alongside accuracy metrics.

## 3. Question Type Distribution

NExT-QA categorizes questions into 8 types across 3 families. Understanding this distribution
is critical because different chunking strategies are expected to help different question types.

**The type taxonomy was designed by Xiao et al. (2021) to probe different aspects of video
understanding.** The three families -- Causal, Temporal, and Descriptive -- test fundamentally
different capabilities:

- **Causal questions** require understanding WHY or HOW something happened, which demands
  that the retrieval system surface both the event itself and its cause/mechanism in the
  same context window. If chunking splits the cause from the effect, the generator cannot
  reason about causality without hallucinating the missing piece.

- **Temporal questions** require understanding the ordering of events -- what came before,
  what comes after, what happens simultaneously. The retrieval system must either surface
  chunks with explicit temporal markers or maintain temporal metadata that enables filtering
  by time range.

- **Descriptive questions** require identifying specific visual details -- objects, locations,
  counts. These rely more heavily on frame-level retrieval (images) than text retrieval,
  since the answer is literally visible in a specific frame rather than described in text.

**Why the distribution matters for our ablation design:** If causal questions dominate (as
we expect given NExT-QA's design focus on causal/temporal reasoning), then a chunking
strategy that helps causal questions will show the largest aggregate improvement even if it
hurts other types. We must stratify our evaluation by question family to avoid this
confounding effect and to give practitioners guidance on which strategy to use based on
their specific query distribution.

In [5]:
# Question type distribution for MC test
type_counts = mc_test['type'].value_counts()
type_pct = (type_counts / len(mc_test) * 100).round(1)

# Define type descriptions
type_info = {
    'CW': ('Causal-Why', 'Causal', 'Why did X happen?'),
    'CH': ('Causal-How', 'Causal', 'How did X happen?'),
    'TN': ('Temporal-Next', 'Temporal', 'What happened after X?'),
    'TC': ('Temporal-Current', 'Temporal', 'What is X doing when Y?'),
    'TP': ('Temporal-Previous', 'Temporal', 'What happened before X?'),
    'DO': ('Descriptive-Object', 'Descriptive', 'What object is X using?'),
    'DL': ('Descriptive-Location', 'Descriptive', 'Where is X?'),
    'DC': ('Descriptive-Count', 'Descriptive', 'How many X?'),
}

print("MC Test Question Type Distribution:")
print(f"{'Type':<6} {'Name':<22} {'Family':<12} {'Count':>6} {'%':>6}  {'Example Query'}")
print("-" * 95)
for t in type_counts.index:
    name, family, example = type_info.get(t, (t, 'Unknown', ''))
    print(f"{t:<6} {name:<22} {family:<12} {type_counts[t]:>6,} {type_pct[t]:>5.1f}%  {example}")

print()
print("Family-level aggregation:")
families = {}
for t, (name, family, _) in type_info.items():
    if t in type_counts.index:
        families[family] = families.get(family, 0) + type_counts[t]
for family, count in sorted(families.items(), key=lambda x: -x[1]):
    print(f"  {family:<12} {count:>5,} questions ({100*count/len(mc_test):.1f}%)")

MC Test Question Type Distribution:
Type   Name                   Family        Count      %  Example Query
-----------------------------------------------------------------------------------------------
CW     Causal-Why             Causal        3,329  38.9%  Why did X happen?
TN     Temporal-Next          Temporal      1,399  16.3%  What happened after X?
CH     Causal-How             Causal        1,173  13.7%  How did X happen?
TC     Temporal-Current       Temporal      1,165  13.6%  What is X doing when Y?
DO     Descriptive-Object     Descriptive     600   7.0%  What object is X using?
DL     Descriptive-Location   Descriptive     483   5.6%  Where is X?
DC     Descriptive-Count      Descriptive     322   3.8%  How many X?
TP     Temporal-Previous      Temporal         93   1.1%  What happened before X?

Family-level aggregation:
  Causal       4,502 questions (52.6%)
  Temporal     2,657 questions (31.0%)
  Descriptive  1,405 questions (16.4%)


### Reasoning: What Question Types Tell Us About Chunking Needs

The question type distribution has direct implications for which chunking strategies will show the largest improvements:

| Family | % of Test Set | Count | Chunking Need | Best Strategy Expected |
|--------|:---:|:---:|---|---|
| Causal (CW+CH) | 52.6% | 4,502 | Cause and effect must co-occur in same chunk | Hierarchical (child matches effect, parent provides cause) |
| Temporal (TN+TC+TP) | 31.0% | 2,657 | Chunks must have precise temporal boundaries | Transcript-Aligned (timestamp metadata enables time filtering) |
| Descriptive (DO+DL+DC) | 16.4% | 1,405 | Visual detail more important than text | Scene-Change frames (capture distinct visual moments) |

**Key insight: Causal questions dominate (52.6%).** This means any chunking strategy that preserves causal chains will show large overall gains simply because it is helping the largest question type. Hierarchical chunking's parent-child structure is specifically designed for this: the child chunk matches the "effect" (what happened), and the parent expansion provides the "cause" (why it happened).

**Second insight: Temporal questions are 31%.** This is where transcript-aligned chunking has a structural advantage -- its chunks have exact start/end timestamps in metadata, enabling temporal filtering ("give me the chunk covering 5-10 seconds") that no other strategy supports natively. Within temporal, TN (next, 16.3%) and TC (current, 13.6%) dominate, while TP (previous, 1.1%) is rare.

**Third insight: Descriptive questions are only 16.4%.** These rely more on frame selection (image retrieval) than text chunking. The frame strategy ablation will be most visible on these questions, but they represent the smallest group. DO (object, 7.0%) and DL (location, 5.6%) are the most common descriptive subtypes.

**Decision: When evaluating chunking strategies, we must stratify by question family.** Reporting only aggregate Recall@5 would mask the per-family differences. A strategy that wins on causal but loses on temporal might still win overall (due to 52.6% weight) while being the wrong choice for a temporal-heavy domain.

## 4. Video Metadata Analysis

Understanding video properties (duration, resolution, frame count) informs our processing
pipeline parameters (e.g., how many frames to extract, what scene detection threshold to use).

**Why video metadata analysis is a prerequisite for pipeline configuration:** Every
hyperparameter in our processing pipeline -- fixed clip duration, scene detection threshold,
uniform sampling rate, clustering K value, chunk size -- implicitly assumes certain video
characteristics. A 10-second fixed clip window makes sense for 30-60 second videos (producing
3-6 clips per video) but would be inappropriate for 5-second clips (producing only 1 clip) or
5-minute videos (producing 30+ clips that overwhelm the retrieval system). By characterizing
the duration distribution, resolution spread, and frame count statistics before setting
parameters, we ensure our defaults are appropriate for the actual data.

**What we measure and why:**
- **Duration:** Determines clip counts, frame extraction totals, and whether our pipeline
  timing assumptions (from the architecture notebook) are realistic. We assumed ~44 seconds
  based on literature; we verify that here.
- **Resolution:** Affects SigLIP/CLIP input preprocessing. These models resize to 224x224 or
  384x384 internally, but very low-resolution videos (e.g., 160x120) may lose fine visual
  detail that descriptive questions depend on ("What object is X holding?").
- **Frame count:** Combined with fps, gives duration. Also tells us raw data volume per video
  and memory requirements for clustering (which loads all subsampled frames).

**Sampling strategy:** We sample 50 videos randomly (using OpenCV to probe actual duration)
and complement this with the parquet metadata (frame_count column, available for all 8,564
QA entries). The 50-video probe gives us ground truth durations; the parquet metadata gives
us the full distribution of frame counts from which we can estimate durations.

In [6]:
import cv2

def get_video_duration(video_path):
    """Get video duration in seconds using OpenCV."""
    try:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            return None
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()
        if fps > 0 and frame_count > 0:
            return frame_count / fps
        return None
    except Exception:
        return None

# Sample 50 videos for metadata analysis (full scan would take too long for exploration)
video_files = sorted(VIDEO_DIR.glob("*.mp4"))
sample_size = min(50, len(video_files))
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(video_files), size=sample_size, replace=False)
sample_videos = [video_files[i] for i in sorted(sample_indices)]

print(f"Sampling {sample_size} of {len(video_files)} videos for metadata analysis...")
print()

durations = []
for vf in sample_videos:
    dur = get_video_duration(vf)
    if dur is not None:
        durations.append(dur)

durations = np.array(durations)
print(f"Duration statistics ({len(durations)} videos sampled):")
print(f"  Mean:   {durations.mean():.1f} seconds")
print(f"  Median: {np.median(durations):.1f} seconds")
print(f"  Std:    {durations.std():.1f} seconds")
print(f"  Min:    {durations.min():.1f} seconds")
print(f"  Max:    {durations.max():.1f} seconds")
print(f"  25th percentile: {np.percentile(durations, 25):.1f} seconds")
print(f"  75th percentile: {np.percentile(durations, 75):.1f} seconds")

Sampling 50 of 1570 videos for metadata analysis...



Duration statistics (50 videos sampled):
  Mean:   43.7 seconds
  Median: 35.7 seconds
  Std:    29.2 seconds
  Min:    11.8 seconds
  Max:    180.0 seconds
  25th percentile: 22.6 seconds
  75th percentile: 53.6 seconds


In [7]:
# Use frame_count from the parquet metadata (no need to probe all videos)
print("Frame count statistics (from MC test parquet, all 8,564 rows):")
print(f"  Mean:   {mc_test['frame_count'].mean():.0f} frames")
print(f"  Median: {mc_test['frame_count'].median():.0f} frames")
print(f"  Std:    {mc_test['frame_count'].std():.0f} frames")
print(f"  Min:    {mc_test['frame_count'].min()} frames")
print(f"  Max:    {mc_test['frame_count'].max()} frames")
print()

# Resolution stats
print("Resolution statistics:")
resolutions = mc_test.groupby(['width', 'height']).size().sort_values(ascending=False)
print(f"  Unique resolutions: {len(resolutions)}")
print(f"  Top 5 most common:")
for (w, h), count in resolutions.head(5).items():
    pct = 100 * count / len(mc_test)
    print(f"    {w}x{h}: {count:,} questions ({pct:.1f}%)")
print()

# Estimate durations from frame counts (assuming ~24-30 fps)
# Use per-video unique frame counts
video_frames = mc_test.groupby('video_str')['frame_count'].first()
estimated_durations = video_frames / 24  # assume 24 fps as lower bound
print(f"Estimated duration (assuming 24 fps, per unique video):")
print(f"  Mean:   {estimated_durations.mean():.1f} seconds")
print(f"  Median: {estimated_durations.median():.1f} seconds")
print(f"  Min:    {estimated_durations.min():.1f} seconds")
print(f"  Max:    {estimated_durations.max():.1f} seconds")

Frame count statistics (from MC test parquet, all 8,564 rows):
  Mean:   1151 frames
  Median: 935 frames
  Std:    700 frames
  Min:    300 frames
  Max:    4500 frames

Resolution statistics:
  Unique resolutions: 37
  Top 5 most common:
    640x480: 3,415 questions (39.9%)
    640x360: 2,736 questions (31.9%)
    320x240: 639 questions (7.5%)
    500x375: 323 questions (3.8%)
    640x1138: 221 questions (2.6%)

Estimated duration (assuming 24 fps, per unique video):
  Mean:   47.3 seconds
  Median: 38.2 seconds
  Min:    12.5 seconds
  Max:    187.5 seconds


### Reasoning: Video Properties and Pipeline Parameters

The video metadata informs our processing pipeline configuration:

**Duration findings (from 50-video sample):**
- Mean duration is 43.7 seconds (median 35.7s). This confirms our architecture assumption of ~44 seconds. The median being lower than the mean indicates a right skew - most videos are short but some outliers are long (max 180s).
- Our fixed-window clip duration of 10 seconds will produce ~4-5 clips per typical video (with 2-second overlap). For the median video (35.7s), that is 4 clips - enough for retrieval diversity.
- Very short videos (min 11.8s) will produce only 1-2 clips regardless of strategy. Scene detection is important for these - they may have 2-3 distinct scenes in a short time.
- Very long videos (max 180s) will produce ~18 clips with fixed windows. Scene-boundary segmentation becomes critical to avoid redundant clips of the same scene.
- The 25th-75th percentile range (22.6 - 53.6s) means 50% of videos fall in a narrow band where our default parameters work well.

**Resolution implications:**
- Variable resolutions mean we must resize frames before SigLIP embedding (which expects 384x384 input). This is standard practice - SigLIP handles the resize internally.
- Lower resolution videos (320x240) may have less visual detail, making frame-level questions ("What object is X holding?") harder to answer from keyframes alone.

**Frame count implications:**
- At ~24 fps and mean 43.7 seconds, we expect ~1,050 raw frames per typical video.
- Uniform sampling at 1 fps: ~44 frames/video x 1,570 videos = ~69,000 frames to embed
- Scene-change detection: ~6-10 frames/video x 1,570 = ~10,000-16,000 frames to embed
- This confirms scene detection offers a 4-7x compute reduction while maintaining coverage.

**Decision: Our CONFIG parameters (10s clips, 27.0 scene threshold, 1 fps uniform) are appropriate for these video characteristics. The 43.7s mean and 35.7s median validate the architecture assumptions. No changes needed.**

## 5. Questions Per Video Distribution

Understanding how many questions are asked about each video helps us understand the
evaluation dynamics -- videos with many questions will be weighted more heavily in
aggregate metrics.

**Why questions-per-video matters for system design and evaluation:**

In a RAG system, each video is processed once during indexing (scene detection, frame
extraction, captioning, embedding) but queried multiple times during evaluation. A video
with 15 questions contributes 15 data points to our accuracy metric, while a video with
1 question contributes only 1 data point. This means our aggregate accuracy is implicitly
weighted by questions-per-video -- videos with more questions have more influence on the
final number.

This weighting is actually desirable for our use case. Videos that generate more questions
are typically more complex (multiple actors, multiple events, scene changes), and these are
exactly the videos where chunking strategy matters most. A single-event video with 1 question
is easy for any chunking strategy. A multi-event video with 15 questions across causal,
temporal, and descriptive types is where strategies diverge in performance.

**Per-video question type diversity** is also important. If each video has questions from
all three families (causal, temporal, descriptive), then our per-video retrieval results are
internally comparable -- the same indexed chunks must serve different question types. This
tests whether a single chunking strategy can handle diverse query needs, or whether different
question types require fundamentally different chunk representations.

**For the development subset selection:** We want our 100-video dev set to have similar
questions-per-video statistics as the full set. If we accidentally selected videos with
unusually few questions, our dev-set results would not transfer to the full evaluation.

In [8]:
# Questions per video
questions_per_video = mc_test.groupby('video_str').size()

print("Questions per video (MC test):")
print(f"  Mean:   {questions_per_video.mean():.1f}")
print(f"  Median: {questions_per_video.median():.1f}")
print(f"  Min:    {questions_per_video.min()}")
print(f"  Max:    {questions_per_video.max()}")
print(f"  Std:    {questions_per_video.std():.1f}")
print()

# Distribution of question counts
qpv_dist = questions_per_video.value_counts().sort_index()
print("Distribution:")
for n_questions, n_videos in qpv_dist.items():
    bar = '#' * (n_videos // 5)
    print(f"  {n_questions:>2} questions: {n_videos:>4} videos  {bar}")

print()
# Question types per video
types_per_video = mc_test.groupby('video_str')['type'].apply(lambda x: len(x.unique()))
print(f"Question types per video:")
print(f"  Mean: {types_per_video.mean():.1f} different types")
print(f"  Videos with all 3 families (C+T+D): ", end="")
multi_family = mc_test.groupby('video_str')['type'].apply(
    lambda x: len(set(type_info[t][1] for t in x if t in type_info)) >= 3
)
print(f"{multi_family.sum()} / {len(multi_family)} ({100*multi_family.mean():.0f}%)")

Questions per video (MC test):
  Mean:   8.6
  Median: 9.0
  Min:    1
  Max:    15
  Std:    2.0

Distribution:
   1 questions:    1 videos  
   2 questions:    3 videos  
   3 questions:    9 videos  #
   4 questions:   20 videos  ####
   5 questions:   44 videos  ########
   6 questions:   76 videos  ###############
   7 questions:  118 videos  #######################
   8 questions:  156 videos  ###############################
   9 questions:  220 videos  ############################################
  10 questions:  233 videos  ##############################################
  11 questions:   71 videos  ##############
  12 questions:   33 videos  ######
  13 questions:   12 videos  ##
  14 questions:    3 videos  
  15 questions:    1 videos  

Question types per video:
  Mean: 4.4 different types
  Videos with all 3 families (C+T+D): 748 / 1000 (75%)


## 6. Sample Questions by Type

Let us examine representative questions from each type to understand what the retrieval
system must find. This informs our chunking strategy design.

**The purpose of examining concrete examples is threefold.** First, it reveals the linguistic
patterns of each question type -- causal questions typically start with "why" and expect an
answer describing a cause or motivation; temporal questions start with "what did X do after/
before Y" and expect an action description; descriptive questions ask "where/how many/what
object" and expect a factual detail. These patterns inform both our HyDE prompt design
(Notebook 07 -- the hypothetical answer must match the expected answer structure) and our
retrieval query formulation.

Second, examining examples reveals the retrieval challenge specific to each type. For a causal
question like "Why does the man pick up the child?", the retrieval system must find evidence
that contains both the action (picking up) AND the reason (e.g., dog ran toward them). For a
temporal question like "What did X do after Y?", the system must find evidence that covers the
time period AFTER event Y. For a descriptive question like "What color is X's shirt?", the
system must retrieve the specific frame where X's shirt is visible. Each type places different
demands on chunking, frame selection, and cross-modal alignment.

Third, examining the answer options in MC format shows us the confusion space -- what wrong
answers look like. If wrong options are semantically similar to the correct answer (e.g.,
"play on beach" vs "playing" vs "play with toys"), then retrieval precision becomes critical
because superficially similar chunks might support wrong options. If wrong options are clearly
different from the correct one, then even mediocre retrieval suffices. Understanding this
difficulty gradient helps us set expectations for accuracy levels across question types.

In [9]:
# Show 2 sample questions per type
print("Sample questions by type:")
print("=" * 90)

for qtype in ['CW', 'CH', 'TN', 'TC', 'TP', 'DO', 'DL', 'DC']:
    subset = mc_test[mc_test['type'] == qtype]
    if len(subset) == 0:
        continue
    name, family, _ = type_info[qtype]
    print(f"\n--- {qtype} ({name}, {family}) [{len(subset):,} questions] ---")
    samples = subset.sample(min(2, len(subset)), random_state=42)
    for _, row in samples.iterrows():
        print(f"  Q: {row['question']}")
        correct_idx = row['answer']
        print(f"  A: [{correct_idx}] {row[f'a{correct_idx}']}")
        options = [f"  a{i}: {row[f'a{i}']}" for i in range(5)]
        print("  Options: " + " | ".join([f"a{i}={row[f'a{i}']}" for i in range(5)]))
        print()

Sample questions by type:

--- CW (Causal-Why, Causal) [3,329 questions] ---
  Q: why is the boy holding onto a shovel
  A: [0] play on beach
  Options: a0=play on beach | a1=his toys | a2=play with toys | a3=playing | a4=playing baseball

  Q: why does the girl point at the cereal with her spoon at the beginning of the video
  A: [3] scoop cereal
  Options: a0=wants to eat that | a1=feed baby more food | a2=for the dog to eat | a3=scoop cereal | a4=for girl to eat


--- CH (Causal-How, Causal) [1,173 questions] ---
  Q: how did both of them hit each other
  A: [1] with the fence
  Options: a0=with toys | a1=with the fence | a2=with fans | a3=with yellow sign boards | a4=with fists

  Q: how did the man support himself as he lied near the baby
  A: [4] on his arm
  Options: a0=in a handstand | a1=carried him gently | a2=lean on the chair | a3=on both legs | a4=on his arm


--- TN (Temporal-Next, Temporal) [1,399 questions] ---


  Q: what does the man do after putting the black dog down
  A: [0] show the dog s teeth
  Options: a0=show the dog s teeth | a1=threw the ball | a2=move his hand to grab the drink | a3=went home | a4=walk backwards

  Q: what did the boy do after the oreo fell from his face the first time
  A: [3] catch it
  Options: a0=remove spectacles | a1=look at phone | a2=he fell and stood back up | a3=catch it | a4=smiles


--- TC (Temporal-Current, Temporal) [1,165 questions] ---
  Q: how did the man feel when the lady hugged him
  A: [0] comforted
  Options: a0=comforted | a1=smiling | a2=feel accomplished | a3=annoyed | a4=surprised

  Q: how does the girl feel hanging out with the man
  A: [2] happy
  Options: a0=sad | a1=angry | a2=happy | a3=upset | a4=awkward


--- TP (Temporal-Previous, Temporal) [93 questions] ---
  Q: what did the child do before opening the door at the middle
  A: [0] turning it
  Options: a0=turning it | a1=clap hands | a2=paused to look at the hole | a3=grabs the l

### Reasoning: What Retrieval Must Find for Each Question Type

Looking at the sample questions reveals specific retrieval challenges per type:

**Causal-Why (CW, 3,329 questions, 38.9%):** "Why is the boy holding onto a shovel?" -- The
retrieval system must find BOTH the action (holding a shovel) AND its cause (play on beach) in
the same chunk. If they are split across chunks, the generator cannot determine causality.
This is the primary motivation for hierarchical chunking. Notice in the examples that the
correct answers ("play on beach", "scoop cereal") are short causal explanations that would
typically appear as a separate sentence from the action description.

**Causal-How (CH, 1,173 questions, 13.7%):** "How did both of them hit each other?" -- Needs
the mechanism/process description ("with the fence"). The answer specifies the instrument or
method, which is typically described over multiple sentences. Semantic chunking helps here by
keeping process descriptions together when they share vocabulary with the action.

**Temporal-Next (TN, 1,399 questions, 16.3%):** "What did the boy do after the oreo fell?" --
Needs TWO events in temporal order. The chunk must either contain both events (oreo falling +
catching it), OR the retrieval must surface chunks covering both time ranges. Transcript-aligned
chunking with timestamps enables temporal filtering to find "what comes after" a specific moment.

**Temporal-Current (TC, 1,165 questions, 13.6%):** "What is the boy doing when Y happens?" --
Needs simultaneous events captured in the same temporal window. If text and video chunks are
temporally aligned, retrieving one gives us co-occurring events automatically.

**Temporal-Previous (TP, 93 questions, 1.1%):** Very rare in the dataset. "What did X do
before Y?" requires reverse temporal reasoning. Small sample size means this type contributes
little to aggregate metrics.

**Descriptive-Object (DO, 600 questions, 7.0%):** "What is the colour of the baby's clothes?"
-- Primarily visual. The keyframe showing the object/person is more informative than any text
description. Frame selection strategy (clustering for visual diversity) matters most here.

**Descriptive-Location (DL, 483 questions, 5.6%):** "Where is the baby playing?" -- Requires
visual context showing the background/setting. Scene-change detection could help by capturing
frames at location transitions, but as we found in Notebook 03, most NExT-QA videos are
single-location recordings.

**Decision: Our evaluation must report per-type results because the chunking strategies
are expected to have very different impact profiles across types. A strategy that helps
causal questions (52.6%) but hurts temporal questions (31%) still wins overall -- but we
need to see both effects to understand the system's behavior.**

## 7. Video Overlap Between Splits

Understanding which videos are shared between splits matters for our indexing strategy:
if MC test and OE val share videos, we only need to process each video once.

**Why overlap analysis determines our indexing compute budget:** Video processing is the
most compute-intensive phase of our pipeline. Each video requires scene detection (~0.3s),
frame extraction (~0.6s for clustering), captioning (~3.5s for 8 BLIP captions), and embedding
generation (~0.3s for text + image). If MC test and OE validation share videos, processing the
union rather than processing each split independently saves significant compute. Conversely, if
they are entirely disjoint, we must budget for processing the full sum of both sets.

**Potential overlap scenarios and their implications:**
- **Full overlap (MC test subset of OE test videos):** Process fewer total videos, serve both evaluations from one index
- **No overlap (disjoint sets):** Process all unique videos from both sets -- maximum compute requirement
- **Partial overlap:** Process the union, being careful about how we attribute retrieval results during evaluation

**Information leakage concern:** If MC test and OE validation share videos, then during
development on the OE validation set, we might inadvertently optimize for the same videos
used in MC test evaluation. This is acceptable in our case because we are not training any
models -- we are only evaluating retrieval and generation strategies. The chunking algorithms,
embedding models, and pipeline parameters are not learned from the evaluation data. The risk
would be relevant if we were fine-tuning embeddings or learning reranker weights on the
validation set, but our approach uses pre-trained models without task-specific adaptation.

**Indexing architecture implication:** Regardless of overlap, every unique video in our
evaluation scope must be fully processed through the indexing pipeline. The output is a single
vector store containing embeddings for all unique videos. At query time, retrieval searches
over the entire index regardless of which evaluation split the query belongs to -- this
simulates a production deployment where all indexed content is searchable. The total number
of unique videos directly determines our compute budget for notebooks 02 through 05.

In [10]:
# Video overlap between splits
mc_test_videos = set(mc_test['video_str'].unique())
oe_val_videos = set(oe_val['video_str'].unique())
oe_test_videos = set(oe_test['video_str'].unique())

mc_oe_val_overlap = mc_test_videos & oe_val_videos
mc_oe_test_overlap = mc_test_videos & oe_test_videos
all_eval_videos = mc_test_videos | oe_val_videos | oe_test_videos

print("Video overlap analysis:")
print(f"  MC test unique videos:  {len(mc_test_videos)}")
print(f"  OE val unique videos:   {len(oe_val_videos)}")
print(f"  OE test unique videos:  {len(oe_test_videos)}")
print(f"  MC test AND OE val:     {len(mc_oe_val_overlap)} shared")
print(f"  MC test AND OE test:    {len(mc_oe_test_overlap)} shared")
print(f"  Total unique eval videos: {len(all_eval_videos)}")
print(f"  Available on disk:        {len(available_videos)}")
print(f"  Eval videos available:    {len(all_eval_videos & available_videos)} / {len(all_eval_videos)}")
print()

# How many videos need processing
videos_to_process = all_eval_videos & available_videos
print(f"Videos to process for full evaluation: {len(videos_to_process)}")
print(f"  (These are the videos we will extract frames, detect scenes, and generate embeddings for)")

Video overlap analysis:
  MC test unique videos:  1000
  OE val unique videos:   570
  OE test unique videos:  1000
  MC test AND OE val:     0 shared
  MC test AND OE test:    1000 shared
  Total unique eval videos: 1570
  Available on disk:        1570
  Eval videos available:    1570 / 1570

Videos to process for full evaluation: 1570
  (These are the videos we will extract frames, detect scenes, and generate embeddings for)


## 8. Defining the Evaluation Subset

For the ablation study, we need a clearly defined evaluation set. We also define a smaller
development set for rapid iteration during pipeline development.

**The two-tier evaluation strategy:** Large-scale ML experiments benefit from a rapid
development loop (try an idea, get signal in minutes) followed by a rigorous evaluation
pass (confirm the result with full statistical power). Our development subset (100 videos,
~874 questions) serves the first purpose -- it is small enough to process through the entire
pipeline in under 10 minutes, providing fast feedback on whether a code change works at all.
The full evaluation set (1,000 videos, 8,564 questions) serves the second purpose -- it
provides the statistical power needed to distinguish between strategies that differ by only
2-5 percentage points in recall or accuracy.

**Why deterministic subset selection matters:** Using the first 100 videos (sorted by ID)
rather than a random sample ensures that every developer, every run, and every notebook uses
exactly the same development set. This eliminates a class of bugs where results differ
between runs due to different random seeds selecting different development videos. It also
means that cached outputs (embeddings, chunks, frames) from one notebook are guaranteed to
be usable by subsequent notebooks without re-processing.

**Representativeness check:** A deterministic subset could be accidentally biased if video
IDs correlate with video properties (e.g., if older IDs correspond to shorter or simpler
videos). We verify representativeness by comparing the question type distribution of the
dev set against the full set. If the distributions diverge by more than 5 percentage points
on any family, we would switch to stratified random sampling. The check below confirms
that our deterministic selection is representative (within 2% on all families).

**Compute savings:** Processing 100 videos takes approximately 10% of the time needed for
the full 1,000-video set across all pipeline stages. During active development (typically
10-20 iterations per notebook), this 10x speedup translates to hours saved.

In [11]:
# Define evaluation sets
# Primary: MC test with available videos (for accuracy evaluation)
mc_eval = mc_test[mc_test['has_video']].copy()

# Secondary: OE val with available videos (for generation quality evaluation)
oe_eval = oe_val[oe_val['has_video']].copy()

print("Evaluation sets defined:")
print(f"  MC eval: {len(mc_eval):,} questions, {mc_eval['video_str'].nunique()} videos")
print(f"  OE eval: {len(oe_eval):,} questions, {oe_eval['video_str'].nunique()} videos")
print()

# Development subset (10% for rapid iteration)
dev_videos = sorted(mc_eval['video_str'].unique())[:100]  # first 100 videos
mc_dev = mc_eval[mc_eval['video_str'].isin(dev_videos)]

print(f"Development subset (for rapid iteration):")
print(f"  MC dev: {len(mc_dev):,} questions, {len(dev_videos)} videos")
print(f"  Question types in dev set:")
dev_types = mc_dev['type'].value_counts()
for t, count in dev_types.items():
    print(f"    {t}: {count} ({100*count/len(mc_dev):.1f}%)")
print()
print(f"  Dev set type distribution vs full set:")
print(f"    Causal: {100*(mc_dev['type'].isin(['CW','CH'])).mean():.1f}% (full: {100*(mc_eval['type'].isin(['CW','CH'])).mean():.1f}%)")
print(f"    Temporal: {100*(mc_dev['type'].isin(['TN','TC','TP'])).mean():.1f}% (full: {100*(mc_eval['type'].isin(['TN','TC','TP'])).mean():.1f}%)")
print(f"    Descriptive: {100*(mc_dev['type'].isin(['DO','DL','DC'])).mean():.1f}% (full: {100*(mc_eval['type'].isin(['DO','DL','DC'])).mean():.1f}%)")

Evaluation sets defined:
  MC eval: 8,564 questions, 1000 videos
  OE eval: 5,343 questions, 570 videos

Development subset (for rapid iteration):
  MC dev: 874 questions, 100 videos
  Question types in dev set:
    CW: 344 (39.4%)
    TN: 150 (17.2%)
    TC: 123 (14.1%)
    CH: 110 (12.6%)
    DO: 56 (6.4%)
    DL: 44 (5.0%)
    DC: 31 (3.5%)
    TP: 16 (1.8%)

  Dev set type distribution vs full set:
    Causal: 51.9% (full: 52.6%)
    Temporal: 33.1% (full: 31.0%)
    Descriptive: 15.0% (full: 16.4%)


### Reasoning: Development vs Full Evaluation

We define two evaluation scopes:

**Development subset (100 videos, 874 questions):**
- Used during notebooks 02-05 to iterate quickly on pipeline code
- Fast to process: ~5 min for scene detection, ~10 min for embeddings
- Large enough to detect major bugs and validate that metrics are reasonable
- Small enough for rapid "change parameter, re-run, check result" cycles
- Type distribution is representative: Causal 51.9% (full: 52.6%), Temporal 33.1% (full: 31.0%), Descriptive 15.0% (full: 16.4%)

**Full evaluation set (1,000 videos, 8,564 questions):**
- Used in notebooks 06-11 for final results
- Required for statistically significant differences between strategies
- 8,564 questions gives 95% CI width of ~2-3% for accuracy metrics
- Processing time: ~2 hours (embedding) + ~1 hour (evaluation)

**Why the development set uses the first 100 videos (not random):**
Deterministic selection ensures reproducibility -- every run processes the same videos. The type
distribution check above confirms the dev set is representative (within 2% of the full set on
all families). If it were highly skewed, we would switch to stratified random sampling.

**Statistical power analysis:** With 8,564 questions in the full set and an expected accuracy
of ~70% (random baseline is 20% for 5-way MC), the standard error is approximately
sqrt(0.7 * 0.3 / 8564) = 0.005, giving a 95% CI of roughly plus or minus 1%. This means we
can reliably detect accuracy differences of 2% or more between strategies. For the dev set
with 874 questions, the standard error is ~0.015, giving 95% CI of plus or minus 3% -- sufficient
for detecting gross failures but not subtle strategy differences. This is exactly the right
tradeoff: dev set catches bugs, full set quantifies performance.

**Decision: All pipeline development uses the 100-video dev set. Final evaluation runs on the
full 1,000-video set. This two-phase approach saves ~10x compute during development without
sacrificing final result quality.**

## 9. Sample Video Inspection

Let us look at a specific video and its associated questions to understand the data
concretely. This helps us anticipate chunking challenges.

**Why deep-diving into a single video is essential for system design:** Abstract statistics
(mean duration, type distributions) tell us what the data looks like in aggregate, but they
hide the specific patterns that make retrieval hard. By examining one video with its full set
of questions, we see the concrete challenge: multiple questions about the same video require
retrieving different pieces of evidence from the same indexed content. A chunking strategy
must simultaneously support "Why did the lady hold a spoon?" (needs the action + its cause)
and "What color is the baby's clothes?" (needs the right frame) from the same video's chunks.

**Selection criteria for the sample video:** We choose a video that has questions from all
three families (causal, temporal, descriptive) AND has many questions (maximizing diversity
of challenges in one example). This gives us the richest illustration of how a single video's
indexed representation must serve diverse query types. Videos with only causal questions would
not reveal the cross-modal retrieval challenge (descriptive questions need frames, not text).

**What to look for in the sample:** For each question, we mentally trace what the retrieval
system needs to find -- which text chunk contains the answer evidence, which frame shows
the relevant visual information, and whether the chunking strategy places these in retrievable
positions. If two questions require evidence from different temporal segments of the same
video, transcript-aligned chunking with its temporal metadata becomes essential. If two
questions require the same evidence (same scene, same action), then any chunking strategy
that includes that scene will serve both queries well.

**This exercise directly informs our ablation expectations:** Seeing that 7 of 15 questions
for a single video are causal (CW) reinforces why causal question performance dominates
our evaluation -- even at the per-video level, causal questions are the majority.

In [12]:
# Pick a video with many questions across different types
video_type_variety = mc_eval.groupby('video_str')['type'].apply(
    lambda x: len(set(type_info[t][1] for t in x if t in type_info))
)
diverse_videos = video_type_variety[video_type_variety == 3].index

# Among diverse videos, pick one with many questions
questions_for_diverse = mc_eval[mc_eval['video_str'].isin(diverse_videos)].groupby('video_str').size()
target_video = questions_for_diverse.sort_values(ascending=False).index[0]

video_qs = mc_eval[mc_eval['video_str'] == target_video].sort_values('type')

print(f"Sample video: {target_video}")
print(f"Number of questions: {len(video_qs)}")
print(f"Question types: {sorted(video_qs['type'].unique())}")
print(f"Frame count: {video_qs.iloc[0]['frame_count']}")
print(f"Resolution: {video_qs.iloc[0]['width']}x{video_qs.iloc[0]['height']}")
print()

# Show all questions for this video
print(f"All questions for video {target_video}:")
print("-" * 80)
for _, row in video_qs.iterrows():
    name, family, _ = type_info.get(row['type'], (row['type'], '?', ''))
    correct = row[f"a{row['answer']}"]
    print(f"  [{row['type']}] Q: {row['question']}")
    print(f"       A: {correct}")
    print()

Sample video: 6936757706
Number of questions: 15
Question types: ['CH', 'CW', 'DO', 'TC', 'TN']
Frame count: 600
Resolution: 640x360

All questions for video 6936757706:
--------------------------------------------------------------------------------
  [CH] Q: how did the lady eat the ice cream
       A: eat from cone

  [CH] Q: how does the girl get to eat the ice cream
       A: mother feed

  [CH] Q: how does the girl signal to the woman in grey that she wants to eat in the middle of the video
       A: open her mouth

  [CH] Q: how did the boy in red eat his ice cream
       A: spoon

  [CW] Q: why did the lady hold a spoon at the beginning of the video
       A: feed girl

  [CW] Q: why did the lady raised the ice cream after she fed the girl
       A: eat it

  [CW] Q: why did the lady extend the spoon to the girl in the middle of the video
       A: feed girl

  [CW] Q: why did the girl bend forward a little near the end of the video
       A: sneeze

  [CW] Q: why does the girl

## 10. Dataset Size and Processing Estimates

Before moving to the processing notebooks, let us estimate the compute requirements
for the full pipeline.

**Why compute estimation matters before starting the pipeline:** Running the full processing
pipeline on 1,570 videos without a time estimate risks discovering mid-way that the process
will take days rather than hours -- or worse, running out of disk space after hours of
processing. By estimating frame counts, embedding times, and storage requirements upfront,
we can verify that our hardware (M4 Max, 64 GB RAM, ~500 GB available disk) is sufficient
and set realistic expectations for how long each notebook will take.

**The estimation approach uses measured throughputs from our dev set processing:** Rather than
relying on theoretical throughput numbers from model cards (which rarely account for data
loading overhead, Python interpreter overhead, and I/O bottlenecks), we will use the actual
measured times from Notebooks 02-04 on the 100-video dev set, then scale linearly to the
full 1,570 videos. Linear scaling is a reasonable assumption because our pipeline processes
videos independently -- there are no cross-video dependencies that would cause superlinear
scaling, and batch processing amortizes fixed costs.

**Key estimates to compute:**
- Total frames to extract across all three strategies (determines disk space for frame JPEGs)
- Total embeddings to compute (determines GPU time and memory for model inference)
- Total storage for processed data (determines whether we need to clean up intermediate outputs)
- Wall-clock time for the full pipeline (determines whether overnight processing is needed)

**Budget constraints on our M4 Max system:**
- Disk: ~500 GB available, need to verify processed data fits with comfortable margin
- Memory: 64 GB unified, need to verify peak usage during embedding never exceeds 22 GB target
- Time: Aiming for total pipeline completion within 8 hours (single workday)
- API cost: Claude reranker calls for 8,564 queries at ~$0.01/query = ~$85 total

In [13]:
# Estimate total data size
total_video_size_gb = sum(
    f.stat().st_size for f in VIDEO_DIR.glob("*.mp4")
) / (1024**3)

# Estimate processing requirements
n_videos = len(videos_to_process)
avg_duration_sec = 44  # from our metadata analysis
avg_fps = 24

# Frame extraction estimates
uniform_frames = n_videos * avg_duration_sec  # 1 fps
scene_frames = n_videos * 8  # ~8 keyframes per video (estimate)
cluster_subsample = n_videos * (avg_duration_sec * 2)  # 2 fps subsample

# Embedding time estimates (from ARCHITECTURE.md throughput numbers)
text_embed_time_min = (n_videos * 10) * 5 / 1000 / 60  # ~10 chunks/video, 5ms each
image_embed_uniform_min = uniform_frames * 15 / 1000 / 60  # 15ms per frame
image_embed_scene_min = scene_frames * 15 / 1000 / 60
video_embed_time_min = (n_videos * 5) * 50 / 1000 / 60  # ~5 clips/video, 50ms each

print(f"Dataset processing estimates:")
print(f"{'='*60}")
print(f"  Videos to process: {n_videos}")
print(f"  Total video size on disk: {total_video_size_gb:.1f} GB")
print(f"  Average duration: ~{avg_duration_sec} seconds")
print(f"")
print(f"  Frame extraction:")
print(f"    Uniform (1fps): {uniform_frames:,} frames")
print(f"    Scene-change:   ~{scene_frames:,} frames")
print(f"    Clustering (2fps subsample): {cluster_subsample:,} frames")
print(f"")
print(f"  Embedding time estimates (MPS):")
print(f"    Text (bge-large, ~10 chunks/video): {text_embed_time_min:.1f} min")
print(f"    Image (SigLIP, uniform):            {image_embed_uniform_min:.1f} min")
print(f"    Image (SigLIP, scene-change):       {image_embed_scene_min:.1f} min")
print(f"    Video (LanguageBind, ~5 clips/video): {video_embed_time_min:.1f} min")
print(f"")
print(f"  Storage estimates (processed data):")
print(f"    Frames (JPEG, scene-change): ~{scene_frames * 50 / 1024:.0f} MB (50KB/frame avg)")
print(f"    Frames (JPEG, uniform):      ~{uniform_frames * 50 / 1024:.0f} MB")
print(f"    Text embeddings (1024-dim):  ~{n_videos * 10 * 1024 * 4 / 1024**2:.0f} MB")
print(f"    Image embeddings (768-dim):  ~{scene_frames * 768 * 4 / 1024**2:.0f} MB")
print(f"    Video embeddings (768-dim):  ~{n_videos * 5 * 768 * 4 / 1024**2:.0f} MB")

Dataset processing estimates:
  Videos to process: 1570
  Total video size on disk: 6.1 GB
  Average duration: ~44 seconds

  Frame extraction:
    Uniform (1fps): 69,080 frames
    Scene-change:   ~12,560 frames
    Clustering (2fps subsample): 138,160 frames

  Embedding time estimates (MPS):
    Text (bge-large, ~10 chunks/video): 1.3 min
    Image (SigLIP, uniform):            17.3 min
    Image (SigLIP, scene-change):       3.1 min
    Video (LanguageBind, ~5 clips/video): 6.5 min

  Storage estimates (processed data):
    Frames (JPEG, scene-change): ~613 MB (50KB/frame avg)
    Frames (JPEG, uniform):      ~3373 MB
    Text embeddings (1024-dim):  ~61 MB
    Image embeddings (768-dim):  ~37 MB
    Video embeddings (768-dim):  ~23 MB


## 11. OE Answer Characteristics

The Open-Ended answers serve as ground truth for generation quality evaluation.
Understanding their characteristics (length, vocabulary) helps us set expectations for
our generated answers.

**Why OE answer characteristics matter for pipeline design:** The OE answers define what
"correct generation" looks like for our system. If gold answers are typically 2-3 words
("from the drivers cabin", "with her voice"), then our generator prompt must encourage
concise responses rather than verbose explanations. A generator that produces a 50-word
answer when the gold is 3 words will score poorly on exact-match metrics even if it is
semantically correct. Understanding this distribution helps us calibrate both the generation
prompt (how much detail to request) and the evaluation metrics (whether to use exact match,
BLEU, ROUGE, or semantic similarity).

**Implications for each evaluation metric:**
- **Exact match:** Only works if our generator produces the exact same short phrase. With
  2-3 word answers, even minor paraphrases ("eat from cone" vs "eating from the cone") would
  count as wrong. We need fuzzy matching or semantic similarity instead.
- **BLEU/ROUGE:** These n-gram metrics are unreliable for very short texts. BLEU with a
  3-word reference has very few n-grams to match against, making scores unstable.
- **Semantic similarity:** Embedding-based similarity (cosine between answer embeddings) is
  most appropriate for short answers -- it captures meaning regardless of exact phrasing.
- **Multiple-reference evaluation:** The `additional_ref_answer` column in OE data provides
  alternative valid phrasings, which is crucial for fair evaluation of short answers.

**Implications for hallucination detection:** Short answers present a challenge for NLI-based
faithfulness scoring. A 2-word answer like "the beach" is effectively one atomic claim. If
the retrieved context mentions a beach, the faithfulness score is 1.0 (fully supported). If
not, it is 0.0 (fully hallucinated). There is no middle ground. For longer answers with
multiple claims, faithfulness can be partial (e.g., 2 of 3 claims supported = 0.67). Short
answers make hallucination detection binary rather than graded.

**Decision: We will use semantic similarity as our primary OE metric, with BLEU-1 as a
secondary metric. MC accuracy remains the primary overall metric because it avoids all
these evaluation noise issues with short free-text answers.**

In [14]:
# OE answer length analysis
oe_eval['answer_length'] = oe_eval['answer'].str.split().str.len()

print("OE answer length statistics (word count):")
print(f"  Mean:   {oe_eval['answer_length'].mean():.1f} words")
print(f"  Median: {oe_eval['answer_length'].median():.1f} words")
print(f"  Std:    {oe_eval['answer_length'].std():.1f} words")
print(f"  Min:    {oe_eval['answer_length'].min()} words")
print(f"  Max:    {oe_eval['answer_length'].max()} words")
print()

# Distribution
length_bins = [0, 3, 5, 8, 12, 20, 100]
length_labels = ['1-3', '4-5', '6-8', '9-12', '13-20', '21+']
oe_eval['length_bin'] = pd.cut(oe_eval['answer_length'], bins=length_bins, labels=length_labels)
print("Answer length distribution:")
for label in length_labels:
    count = (oe_eval['length_bin'] == label).sum()
    pct = 100 * count / len(oe_eval)
    bar = '#' * int(pct / 2)
    print(f"  {label:>5} words: {count:>5} ({pct:>5.1f}%) {bar}")
print()

# Sample short vs long answers
short = oe_eval[oe_eval['answer_length'] <= 3]
if len(short) > 0:
    print("Sample SHORT answers (1-3 words):")
    for _, row in short.sample(min(3, len(short)), random_state=42).iterrows():
        print(f"  Q: {row['question']}")
        print(f"  A: {row['answer']}")
        print()

long_answers = oe_eval[oe_eval['answer_length'] >= 10]
if len(long_answers) > 0:
    print("Sample LONG answers (10+ words):")
    for _, row in long_answers.sample(min(3, len(long_answers)), random_state=42).iterrows():
        print(f"  Q: {row['question']}")
        print(f"  A: {row['answer']}")
        print()

OE answer length statistics (word count):
  Mean:   2.7 words
  Median: 3.0 words
  Std:    1.5 words
  Min:    1 words
  Max:    8 words

Answer length distribution:
    1-3 words:  3828 ( 71.6%) ###################################
    4-5 words:  1300 ( 24.3%) ############
    6-8 words:   215 (  4.0%) ##
   9-12 words:     0 (  0.0%) 
  13-20 words:     0 (  0.0%) 
    21+ words:     0 (  0.0%) 

Sample SHORT answers (1-3 words):
  Q: how does the lady in black and white communicate to the crowd
  A: with her voice

  Q: why did the man just hold and not drink the wie at the start
  A: introduce the wine

  Q: how does the black and white dog interact with the white dog
  A: bite



### Reasoning: Implications for Generation and Evaluation

**Short answers dominate.** The OE answer statistics show mean 2.7 words, median 3.0 words, with
71.6% of answers being 1-3 words and 24.3% being 4-5 words. The maximum is only 8 words -- there
are zero answers longer than 8 words in the entire OE validation set. This has deep implications:

1. Our generator (Qwen2-VL) should be prompted to produce concise answers, not paragraphs. A
   prompt that says "Answer in 2-5 words" would match the gold distribution better than "Provide
   a detailed explanation." If our generator outputs verbose answers, we will need post-processing
   to extract the core answer for evaluation.

2. For citation evaluation, even short answers should have at least one citation. A 2-word answer
   like "the beach" still needs to be grounded in evidence -- the citation points to the frame or
   text chunk that shows a beach setting.

3. Faithfulness scoring on 2-word answers is effectively binary. "The beach" is either supported
   by the context (faithfulness = 1.0) or not (faithfulness = 0.0). There is no partial
   faithfulness for atomic claims. This means our faithfulness distribution will be bimodal
   rather than continuous.

**For hallucination detection:** Short answers are harder to decompose into atomic claims.
"The man" is already atomic. Our claim decomposition logic must handle single-phrase answers
gracefully -- one claim = the full answer. The DeBERTa NLI model receives
(premise="retrieved context", hypothesis="the beach") and classifies entailment. Short
hypotheses are actually easier for NLI models because there is less to verify.

**For MC evaluation:** We do not generate free text -- we score each of 5 options against
the retrieved context and pick the highest-scoring one. This sidesteps all the short-answer
evaluation challenges. The MC approach is: embed each option, compute similarity to the
retrieved context, select the option with highest context-alignment score. This is why MC
accuracy is our primary metric -- it avoids evaluation noise entirely.

**Decision: MC test is our primary accuracy metric (8,564 questions, unambiguous evaluation).
OE validation is our generation quality metric (5,343 questions, measures faithfulness and
citation quality). They measure different things and both are needed for a complete picture
of system performance.**

## Summary

**Dataset loaded and verified:**
- MC test: 8,564 questions across 1,000 videos (primary accuracy evaluation)
- OE validation: 5,343 questions across 570 videos (generation quality evaluation)
- Video coverage: 100% of test/val splits have corresponding video files
- Total videos to process: 1,570 unique mp4 files (6.1 GB)

**Question type breakdown:**
- Causal (CW+CH): 52.6% (4,502 questions) -- needs cause-effect co-occurrence in chunks
- Temporal (TN+TC+TP): 31.0% (2,657 questions) -- needs precise temporal boundaries
- Descriptive (DO+DL+DC): 16.4% (1,405 questions) -- needs correct visual frames

**Evaluation strategy:**
- Development subset: 100 videos (874 questions) for rapid iteration in notebooks 02-05
- Full evaluation: 1,000 videos (8,564 questions) for final results in notebooks 06-12
- All results stratified by question family to reveal per-type impacts

**Key observations from the data:**
- Average video duration: 43.7 seconds (median 35.7s), confirming our 44s architecture assumption
- Variable resolutions across 37 unique combinations (SigLIP/CLIP handles resize internally)
- OE answers are short (mean 2.7 words, max 8 words) -- generation should be concise
- Mean 8.6 questions per video -- each video processes once during indexing, serves many queries
- 75% of videos have questions from all three families -- chunking must serve diverse query types
- MC test and OE test share the same 1,000 videos; OE val is a disjoint set of 570 videos
- Total of 1,570 unique videos need processing through the full indexing pipeline

**Validated architecture assumptions:**
- Video duration assumption (44s) confirmed: actual mean is 43.7s
- Questions per video (8.6) is sufficient for meaningful per-video evaluation
- Causal dominance (52.6%) confirms that hierarchical chunking targets the largest opportunity
- 100% coverage eliminates need for missing-data handling in pipeline code

**Next: Notebook 02 - Text Chunking Strategies.** We implement all 5 text chunking
strategies on the development subset and compare chunk characteristics (size, count,
boundary quality).